### Layer 3

In [4]:
import zoneinfo
import pytz

"""
What does zoneinfo know about?

available_timezones() returns a SET of every zone name that resolves to a file
on this machine. That includes canonical zones and backward-compat links
(Asia/Calcutta, US/Eastern, etc..). So it's "What's loadable here",
not the "canon"
"""
def zoneinfo_zones():
    return sorted(zoneinfo.available_timezones())

"""
pytz ships its own frozen copy of the tz database. all_timezones is a flat list
of every name it knows, aliases included
"""
def pytz_zones():
    return sorted(pytz.all_timezones)

# How do the lists differ
def compare_lists():
    zi = set(zoneinfo_zones())
    pz = set(zoneinfo_zones())

    both = zi & pz
    only_zi = zi - pz
    only_pz = pz -zi

    print(f"zoneinfo count: {len(zi)}")
    print(f"pytz count : {len(pz)}")
    print(f"in both: {len(both)}")
    print(f"only in zoneinfo : {len(only_zi)}")
    print(f"only in pytz : {len(only_pz)}")

    if only_zi:
        print(f"eg. only zoneinfo: {sorted(only_zi)[:5]}")
    if only_pz:
        print(f"eg. only pytz : {sorted(only_pz)[:5]}")


# Is the given string a valid zone?
"""
We catch 3 exceptions because loading can fail in three ways:
 ZoneInfoNotFoundError -> name isn't in the database
 ValueError -> name is malformed (path traversal, absolute)
 OSError -> file exists but its unreadable
"""
def is_valid_zone(name):
    if not isinstance(name, str) or not name:
        return False
    try:
        zoneinfo.ZoneInfo(name)
        return True
    except (zoneinfo.ZoneInfoNotFoundError, ValueError, OSError):
        return False

# main
def main():
    print("-"*60)
    print("1. Zone Counts")
    compare_lists()

    print()

    print("-"*60)
    print("2. validation tests")
    samples = [
       "Asia/Kolkata",       # canonical, valid
        "asia/kolkata",       # wrong case -> invalid (zoneinfo is case-sensitive)
        "Asia/Calcutta",      # deprecated alias -> still valid (link exists)
        "US/Eastern",         # backward link -> still valid
        "Mars/Olympus_Mons",  # nonsense -> invalid
        "",                   # empty -> invalid
        "../etc/passwd",      # path traversal -> invalid (zoneinfo rejects it)
        "Etc/GMT+5",
        "UTC"
    ]


    for name in samples:
        result = is_valid_zone(name)
        print(f" {name!r:24} -> {result}")

if __name__ == "__main__":
    main()

------------------------------------------------------------
1. Zone Counts
zoneinfo count: 599
pytz count : 599
in both: 599
only in zoneinfo : 0
only in pytz : 0

------------------------------------------------------------
2. validation tests
 'Asia/Kolkata'           -> True
 'asia/kolkata'           -> True
 'Asia/Calcutta'          -> True
 'US/Eastern'             -> True
 'Mars/Olympus_Mons'      -> False
 ''                       -> False
 '../etc/passwd'          -> False
 'Etc/GMT+5'              -> True
 'UTC'                    -> True


In [5]:
import os
print(os.name)

nt


In [6]:
from datetime import datetime
from zoneinfo import ZoneInfo
dt = datetime(2026, 1, 1, 12, 0, tzinfo=ZoneInfo("Etc/GMT+5"))
print(dt.utcoffset())   # <- what do you expect?

-1 day, 19:00:00


In [7]:
import random

def print_random_zones(n=10):
    rng = random.Random(42)          # fixed seed = reproducible
    for name in rng.sample(zoneinfo_zones(), n):
        print(f"  {name}")

print_random_zones()

  America/Fort_Wayne
  Africa/Johannesburg
  Asia/Kathmandu
  Asia/Beirut
  Antarctica/McMurdo
  America/Kentucky/Louisville
  America/Dawson
  Pacific/Pago_Pago
  America/Caracas
  Europe/Belgrade


### Layer 4

In [9]:
"""
Layer 4 — Offset & DST Computation
==================================
Given an IANA zone name and an instant, return everything the user needs
to know about that zone at that moment. Then do conversions.
"""

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

UTC = timezone.utc


# ---------------------------------------------------------------------------
# HELPER: format a timedelta as a human-readable UTC offset
# ---------------------------------------------------------------------------
# Why do we need this? Python's timedelta repr is confusing:
#     timedelta(hours=-5) prints as "-1 day, 19:00:00"
# We want it to print as "-05:00".
#
# Why not just use str()? Because str() of a timedelta gives you "5:30:00"
# for the offset — fine for reading, terrible for tables, and it silently
# misformats negative offsets.
def fmt_offset(delta):
    if delta is None:
        return "n/a"
    total_seconds = int(delta.total_seconds())
    sign = "+" if total_seconds >= 0 else "-"
    total_seconds = abs(total_seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    # Some historical offsets have seconds (e.g. LMT for old cities).
    # We only print them if they're nonzero.
    if seconds:
        return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"
    return f"{sign}{hours:02d}:{minutes:02d}"


def fmt_delta(delta):
    """Format a DST delta as +1h00m, +0h30m, etc."""
    if delta is None:
        return "n/a"
    total_seconds = int(delta.total_seconds())
    sign = "+" if total_seconds >= 0 else "-"
    total_seconds = abs(total_seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, _ = divmod(remainder, 60)
    return f"{sign}{hours}h{minutes:02d}m"


# ---------------------------------------------------------------------------
# CORE: snapshot a zone at a given instant
# ---------------------------------------------------------------------------
# This is the heart of Layer 4. Everything else is a wrapper around this.
#
# The key identity:
#     total_offset == standard_offset + dst_delta
#
# zoneinfo gives us the total (via utcoffset) and the DST delta (via dst),
# so standard_offset is just the difference. NOTHING is hardcoded.
def zone_snapshot(zone_name, moment=None):
    tz = ZoneInfo(zone_name)

    # Default to "now" if no moment given.
    if moment is None:
        moment = datetime.now(UTC)

    # If someone passed a naive datetime, assume they meant UTC.
    # This is a convention, not magic — the caller should pass aware
    # datetimes whenever possible.
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=UTC)

    # Convert to UTC, then to the target zone. Two steps, always in this
    # order. The instant never changes; only the wall-clock representation.
    instant_utc = moment.astimezone(UTC)
    local = instant_utc.astimezone(tz)

    offset = local.utcoffset()                    # total offset
    dst_delta = local.dst() or timedelta(0)       # DST component
    std_offset = offset - dst_delta               # derived

    return {
        "zone": zone_name,
        "utc": instant_utc,
        "local": local,
        "abbrev": local.tzname(),
        "offset": offset,
        "offset_str": fmt_offset(offset),
        "std_offset": std_offset,
        "std_offset_str": fmt_offset(std_offset),
        "dst_delta": dst_delta,
        "dst_delta_str": fmt_delta(dst_delta),
        "dst_active": dst_delta != timedelta(0),
    }


# ---------------------------------------------------------------------------
# CONVERSION: from one zone to another
# ---------------------------------------------------------------------------
# Take a naive local time string, attach the source zone, convert.
#
# The input is a STRING like "2026-03-21 09:00" because that's what a user
# types. We parse it to a naive datetime, then attach the source zone.
def convert_time(local_str, from_zone, to_zone):
    naive = datetime.fromisoformat(local_str)
    src = ZoneInfo(from_zone)
    dst = ZoneInfo(to_zone)

    aware = naive.replace(tzinfo=src)    # attach source zone
    converted = aware.astimezone(dst)    # same instant, new zone

    return {
        "source_local": aware,
        "source_offset": aware.utcoffset(),
        "source_abbrev": aware.tzname(),
        "source_utc": aware.astimezone(UTC),
        "target_local": converted,
        "target_offset": converted.utcoffset(),
        "target_abbrev": converted.tzname(),
    }


# ---------------------------------------------------------------------------
# TEST BLOCK 1: snapshot several zones
# ---------------------------------------------------------------------------
# These are the zones from your brief. Each one is interesting for a
# different reason — read the note next to each.
TEST_ZONES = [
    ("America/New_York",    "DST-observing, US rules"),
    ("Asia/Kolkata",        "half-hour offset, no DST"),
    ("Asia/Kathmandu",      "45-minute offset, no DST"),
    ("Pacific/Chatham",     "45-minute offset, DST-observing"),
    ("Pacific/Kiritimati",  "UTC+14"),
    ("Pacific/Midway",      "UTC-11"),
    ("Antarctica/Troll",    "DST jump is +2 hours, not +1"),
    ("Europe/London",       "BST/GMT"),
    ("Australia/Lord_Howe", "DST shift is only 30 minutes"),
    ("Asia/Tehran",         "DST abolished in 2022"),
]


def print_zone_table():
    print(f"  {'zone':<24} {'local time':<17} {'offset':>7} {'std':>7} "
          f"{'DST':>7}  {'on':<5} {'abbr':<5}")
    print(f"  {'-'*24} {'-'*17} {'-'*7} {'-'*7} {'-'*7}  {'-'*5} {'-'*5}")

    for name, _note in TEST_ZONES:
        s = zone_snapshot(name)
        print(f"  {name:<24} {s['local']:%Y-%m-%d %H:%M}  "
              f"{s['offset_str']:>7} {s['std_offset_str']:>7} "
              f"{s['dst_delta_str']:>7}  {str(s['dst_active']):<5} "
              f"{s['abbrev']:<5}")


# ---------------------------------------------------------------------------
# TEST BLOCK 2: conversions
# ---------------------------------------------------------------------------
def print_conversion(local_str, from_zone, to_zone, note=""):
    r = convert_time(local_str, from_zone, to_zone)
    print(f"\n  {local_str} {from_zone}")
    print(f"      -> {to_zone}")
    if note:
        print(f"      ({note})")
    print(f"      source : {r['source_local']:%Y-%m-%d %H:%M:%S} "
          f"{fmt_offset(r['source_offset'])} {r['source_abbrev']}")
    print(f"      UTC    : {r['source_utc']:%Y-%m-%d %H:%M:%S} +00:00")
    print(f"      target : {r['target_local']:%Y-%m-%d %H:%M:%S} "
          f"{fmt_offset(r['target_offset'])} {r['target_abbrev']}")


# ---------------------------------------------------------------------------
# TEST BLOCK 3: DST transition edge case
# ---------------------------------------------------------------------------
def walk_transition_day(zone_name, year, month, day, hours):
    """Walk through a transition day hour-by-hour, show offsets change."""
    tz = ZoneInfo(zone_name)
    print(f"\n  {zone_name} on {year}-{month:02d}-{day:02d}:")
    print(f"      {'local':<8} {'offset':>7}  {'abbr':<5}  {'UTC':<6}")
    for h in hours:
        local = datetime(year, month, day, h, 30, tzinfo=tz)
        utc = local.astimezone(UTC)
        print(f"      {h:02d}:30    {fmt_offset(local.utcoffset()):>7}  "
              f"{local.tzname():<5}  {utc:%H:%M}")


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    print("=" * 70)
    print("1. Zone snapshots (current instant)")
    print("=" * 70)
    print_zone_table()

    print()
    print("=" * 70)
    print("2. Conversions")
    print("=" * 70)
    print_conversion(
        "2026-03-21 09:00",
        "America/New_York",
        "Asia/Tokyo",
        "US on EDT, Japan no DST",
    )
    print_conversion(
        "1950-06-15 12:00",
        "America/Chicago",
        "Europe/Paris",
        "historical — Chicago on CDT, France had no DST 1945-1976",
    )

    print()
    print("=" * 70)
    print("3. DST spring-forward (America/New_York, 2026-03-08)")
    print("=" * 70)
    print("  02:00 EST -> 03:00 EDT. The 02:xx hour does not exist.")
    walk_transition_day("America/New_York", 2026, 3, 8, range(0, 6))

    print()
    print("=" * 70)
    print("4. DST fall-back (America/New_York, 2026-11-01)")
    print("=" * 70)
    print("  02:00 EDT -> 01:00 EST. The 01:xx hour happens twice.")
    walk_transition_day("America/New_York", 2026, 11, 1, range(0, 4))


if __name__ == "__main__":
    main()

1. Zone snapshots (current instant)
  zone                     local time         offset     std     DST  on    abbr 
  ------------------------ ----------------- ------- ------- -------  ----- -----
  America/New_York         2026-10-09 01:59   -04:00  -05:00  +1h00m  True  EDT  
  Asia/Kolkata             2026-10-09 11:29   +05:30  +05:30  +0h00m  False IST  
  Asia/Kathmandu           2026-10-09 11:44   +05:45  +05:45  +0h00m  False +0545
  Pacific/Chatham          2026-10-09 19:44   +13:45  +12:45  +1h00m  True  +1345
  Pacific/Kiritimati       2026-10-09 19:59   +14:00  +14:00  +0h00m  False +14  
  Pacific/Midway           2026-10-08 18:59   -11:00  -11:00  +0h00m  False SST  
  Antarctica/Troll         2026-10-09 07:59   +02:00  +00:00  +2h00m  True  +02  
  Europe/London            2026-10-09 06:59   +01:00  +00:00  +1h00m  True  BST  
  Australia/Lord_Howe      2026-10-09 16:59   +11:00  +10:30  +0h30m  True  +11  
  Asia/Tehran              2026-10-09 09:29   +03:30  +03:30  

In [11]:
def fold_demo():
    print("\n  --- fold on an AMBIGUOUS time (fall-back) ---")
    tz = ZoneInfo("America/New_York")
    for fold in (0, 1):
        dt = datetime(2026, 11, 1, 1, 30, tzinfo=tz, fold=fold)
        print(f"      fold={fold}: {dt}  utcoffset={dt.utcoffset()} "
              f"-> {dt.astimezone(UTC):%H:%M} UTC")

    print("\n  --- fold on a NONEXISTENT time (spring-forward) ---")
    for fold in (0, 1):
        dt = datetime(2026, 3, 8, 2, 30, tzinfo=tz, fold=fold)
        print(f"      fold={fold}: {dt}  utcoffset={dt.utcoffset()} "
              f"-> {dt.astimezone(UTC):%H:%M} UTC")

fold_demo()


  --- fold on an AMBIGUOUS time (fall-back) ---
      fold=0: 2026-11-01 01:30:00-04:00  utcoffset=-1 day, 20:00:00 -> 05:30 UTC
      fold=1: 2026-11-01 01:30:00-05:00  utcoffset=-1 day, 19:00:00 -> 06:30 UTC

  --- fold on a NONEXISTENT time (spring-forward) ---
      fold=0: 2026-03-08 02:30:00-05:00  utcoffset=-1 day, 19:00:00 -> 07:30 UTC
      fold=1: 2026-03-08 02:30:00-04:00  utcoffset=-1 day, 20:00:00 -> 06:30 UTC


# Layer 2b: Coordinates → IANA Zone

Given (lat, lon), return the IANA zone name. Offline, deterministic,
uses polygon boundaries — not latitude strips.

Library: timezonefinder

In [ ]:
# Run once. In a notebook, "!" runs a shell command.
# !pip install timezonefinder

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/53.1 MB ? eta -:--:--
     ---------------------------------------- 0.5/53.1 MB 4.2 MB/s eta 0:00:13
     - -------------------------------------- 1.6/53.1 MB 4.6 MB/s eta 0:00:12
     - -------------------------------------- 2.6/53.1 MB 4.9 MB/s eta 0:00:11
     -- ------------------------------------- 3.7/53.1 MB 5.1 MB/s eta 0:00:10
     --- ------------------------------------ 4.7/53.1 MB 5.0 MB/s eta 0:00:10
     ---- ----------------------------------- 6.0/53.1 MB 5.2 MB/s eta 0:00:10
     ----- ---------------------------------- 7.3/53.1 MB 5.3 MB/s eta 0:00:09
     ------ --------------------------------- 8.4/53.1 MB 5.3 MB/s eta 0:00:09
     ------- -------------------------------- 9.4/53.1 MB 5.3 MB/s eta 0:00:09
     ------- -------------------------------- 10.5/53.1 MB 5.4 MB/s eta 0:00:08
     -------- ------------------------------- 11.8/53.1 MB 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
from timezonefinder import TimezoneFinder

tf = TimezoneFinder()   # loads the polygon data into memory; takes ~0.5s

In [3]:
points = [
    ("Mumbai",     19.0760,  72.8777),
    ("London",     51.5074,  -0.1278),
    ("New York",   40.7128, -74.0060),
    ("Tokyo",      35.6762, 139.6503),
    ("Sydney",    -33.8688, 151.2093),
]

for name, lat, lon in points:
    zone = tf.timezone_at(lat=lat, lng=lon)
    print(f"{name:<10} ({lat:>8.4f}, {lon:>9.4f}) -> {zone}")

Mumbai     ( 19.0760,   72.8777) -> Asia/Kolkata
London     ( 51.5074,   -0.1278) -> Europe/London
New York   ( 40.7128,  -74.0060) -> America/New_York
Tokyo      ( 35.6762,  139.6503) -> Asia/Tokyo
Sydney     (-33.8688,  151.2093) -> Australia/Sydney


In [4]:
edge_cases = [
    ("North Pole",       90.0,    0.0),
    ("South Pole",      -90.0,    0.0),
    ("Null Island",       0.0,    0.0),   # 0,0 — Gulf of Guinea, ocean
    ("Mid Pacific",       0.0, -140.0),
    ("Mid Atlantic",     30.0,  -40.0),
    ("Kashmir border",   34.0,   75.0),
    ("Exactly on line",  51.5000, -0.1246),  # Greenwich Prime Meridian
]

for name, lat, lon in edge_cases:
    zone = tf.timezone_at(lat=lat, lng=lon)
    print(f"{name:<20} -> {zone!r}")

North Pole           -> 'Etc/GMT'
South Pole           -> 'Antarctica/McMurdo'
Null Island          -> 'Etc/GMT'
Mid Pacific          -> 'Etc/GMT+9'
Mid Atlantic         -> 'Etc/GMT+3'
Kashmir border       -> 'Asia/Kolkata'
Exactly on line      -> 'Europe/London'


In [5]:
lat, lon = 40.7128, -74.0060   # New York, near water on the east

print("timezone_at        :", tf.timezone_at(lat=lat, lng=lon))
print("timezone_at_land   :", tf.timezone_at_land(lat=lat, lng=lon))
print("certain_timezone_at:", tf.certain_timezone_at(lat=lat, lng=lon))

timezone_at        : America/New_York
timezone_at_land   : America/New_York
certain_timezone_at: America/New_York


In [6]:
# Just south of the US/Canada border near Niagara
lat, lon = 43.0, -79.0
print("timezone_at        :", tf.timezone_at(lat=lat, lng=lon))
print("timezone_at_land   :", tf.timezone_at_land(lat=lat, lng=lon))
print("certain_timezone_at:", tf.certain_timezone_at(lat=lat, lng=lon))

timezone_at        : America/New_York
timezone_at_land   : America/New_York
certain_timezone_at: America/New_York


## The three methods, summarized

- timezone_at(lat, lng)         → best guess; includes Etc/GMT±N for oceans
- timezone_at_land(lat, lng)    → None if over water; else best guess
- certain_timezone_at(lat, lng) → None near borders; use when precision matters

For a World Clock app: use `timezone_at` and accept the guess.
For a "legal billing" app: use `certain_timezone_at` and refuse ambiguous points.

In [7]:
from zoneinfo import ZoneInfo
from zoneinfo import ZoneInfoNotFoundError

def is_valid_zone(name):
    if not name:
        return False
    try:
        ZoneInfo(name)
        return True
    except (ZoneInfoNotFoundError, ValueError, OSError):
        return False

# Sweep a grid of points, validate every result
import random
rng = random.Random(42)
valid_count = 0
invalid = []
for _ in range(200):
    lat = rng.uniform(-85, 85)
    lon = rng.uniform(-180, 180)
    zone = tf.timezone_at(lat=lat, lng=lon)
    if zone and is_valid_zone(zone):
        valid_count += 1
    else:
        invalid.append((lat, lon, zone))

print(f"{valid_count}/200 random coordinates produced a valid IANA zone")
if invalid:
    print("Failures:")
    for lat, lon, zone in invalid[:5]:
        print(f"  ({lat:.2f}, {lon:.2f}) -> {zone!r}")

200/200 random coordinates produced a valid IANA zone
